# Device · Aggiornamento e gruppi

`device-da-aggiornare` ha tre **proprietà** in ADR: produttore, modello e **software revision**. Un aggiornamento cambia la software revision; un **gruppo** (una query sulle proprietà) dice quali device ne fanno parte.
Il notebook prova sul servizio vero **due sintassi** della query di un gruppo (quella dell'aiuto della CLI e quella della documentazione), poi cambia la software revision e guarda cosa fa il gruppo.

**Serve:** i notebook 01 e 02a-02c già eseguiti. **Il job vero è nei notebook 14c e 14d.**
**Da dire.** "La software revision è il valore che l'aggiornamento cambia. Il gruppo è una fotografia: si ricalcola alla creazione e a ogni refresh, che si può chiedere una volta all'ora." 

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

In [ ]:
$id = 'device-da-aggiornare'
$modello = 'RobotV1'; $v1 = '1.0.0'; $v2 = '2.0.0'
$sintassi = @(                                         # le query da provare: l'aiuto della CLI (properties.*) e la documentazione (WHERE ...)
    @{ Nome = 'wpc-g-cli';  Query = "properties.model = '$modello' AND properties.softwareRevision = '$v1'" },
    @{ Nome = 'wpc-g-docs'; Query = "WHERE model = '$modello' AND softwareRevision = '$v1'" }
)

## 1 · Il device com'è
Le tre proprietà (produttore, modello, software revision), come le ha scritte il notebook 02c con `az iot adr ns device update`. **Un gruppo le usa nella query**: se mancano, il gruppo resta vuoto. Per questo la cella si ferma se non ci sono.

In [ ]:
$stato = az iot adr ns device show --name $id --namespace $adrName --resource-group $rg --query '{device:name, produttore:properties.manufacturer, modello:properties.model, softwareRevision:properties.softwareRevision}' -o json --only-show-errors | ConvertFrom-Json
$stato | Format-List | Out-String
if (-not $stato.softwareRevision) { throw "Il device $id non ha le proprieta': esegui la cella 'Chi e' il device' del notebook 02c, poi riesegui questo notebook." }
"Se la software revision non e' $v1, riporta il device a $v1 (celle in fondo)." 


device           : device-da-aggiornare
modello          : RobotV1
produttore       : Contoso
softwareRevision : 1.0.0



Se la software revision non e' 1.0.0, riporta il device a 1.0.0 (celle in fondo).

## 2 · Un gruppo con una query
`az iot adr ns group create --query-string …`: la query è **immutabile** dopo la creazione. Si provano due sintassi: il servizio vuole `*` oppure **una clausola `WHERE`**, quindi `WHERE model = … AND softwareRevision = …` è accettata e la forma `properties.model = …` dell'aiuto della CLI è rifiutata (`InvalidGroupQueryFilter`). I gruppi creati qui si cancellano nell'ultima cella.

In [ ]:
# Si parte pulito: un gruppo rimasto da una prova precedente non vale (e il nome sarebbe gia' preso)
foreach ($q in $sintassi) { Remove-AdrGroup -Name $q.Nome -Namespace $adrName -ResourceGroup $rg }
$creati = @()
foreach ($q in $sintassi) {
    "--- $($q.Nome): $($q.Query)"
    $errore = New-AdrGroup -Name $q.Nome -Query $q.Query -Namespace $adrName -ResourceGroup $rg
    if ($errore) { "  RIFIUTATA: $errore"; continue }
    $creati += $q.Nome
    "  accettata. Membri: $((Get-AdrGroupMembers -Name $q.Nome -Namespace $adrName -ResourceGroup $rg) -join ', ')"
}
"gruppi creati: $($creati -join ', ')" 

--- wpc-g-cli: properties.model = 'RobotV1' AND properties.softwareRevision = '1.0.0'

  RIFIUTATA: ERROR: (InvalidGroupQueryFilter) Group query filter (properties.queryFilter) must be '*' or a single WHERE clause.
Code: InvalidGroupQueryFilter
Message: Group query filter (properties.queryFilter) must be '*' or a single WHERE clause.

--- wpc-g-docs: WHERE model = 'RobotV1' AND softwareRevision = '1.0.0'

  accettata. Membri: robot-02, robot-01, device-da-aggiornare

gruppi creati: wpc-g-docs

## 3 · La software revision cambia
È quello che farebbe il device (o il servizio di aggiornamento) dopo un aggiornamento riuscito: scrive la nuova versione in ADR. Il gruppo, però, è una **fotografia**: finché non si rinfresca, non se ne accorge.

In [ ]:
Set-AdrDeviceProperties -Name $id -Namespace $adrName -ResourceGroup $rg -SoftwareRevision $v2
az iot adr ns device show --name $id --namespace $adrName --resource-group $rg --query '{device:name, softwareRevision:properties.softwareRevision}' -o table --only-show-errors
foreach ($n in $creati) {
    "$n : membri ora: $((Get-AdrGroupMembers -Name $n -Namespace $adrName -ResourceGroup $rg) -join ', ')   (conteggio: $(az iot adr ns group count --name $n --namespace $adrName --resource-group $rg -o tsv --only-show-errors 2>&1))"
}

Device                SoftwareRevision

--------------------  ------------------

device-da-aggiornare  2.0.0

wpc-g-docs : membri ora: robot-02, robot-01, device-da-aggiornare   (conteggio: 3)

## 4 · Refresh e pulizia
`group refresh` ricalcola la membership (al massimo **una volta all'ora**); la cella prova, e poi **cancella i gruppi creati qui** e riporta il device alla versione di partenza.

In [ ]:
foreach ($n in $creati) {
    $errore = Update-AdrGroup -Name $n -Namespace $adrName -ResourceGroup $rg
    if ($errore) { "$n : refresh rifiutato: $errore" }
    else { Start-Sleep -Seconds 20; "$n : dopo il refresh i membri sono: $((Get-AdrGroupMembers -Name $n -Namespace $adrName -ResourceGroup $rg) -join ', ')" }
}

wpc-g-docs : refresh rifiutato: ERROR: (GroupRefreshRateLimited) Group membership can only be refreshed once per hour. Please try again later.
Code: GroupRefreshRateLimited
Message: Group membership can only be refreshed once per hour. Please try again later.

## Pulizia
Si cancellano i gruppi creati qui e si riporta il device alla sua software revision, così il notebook si può rilanciare.

In [ ]:
foreach ($q in $sintassi) { Remove-AdrGroup -Name $q.Nome -Namespace $adrName -ResourceGroup $rg; "gruppo $($q.Nome) cancellato (se c'era)" }
Set-AdrDeviceProperties -Name $id -Namespace $adrName -ResourceGroup $rg -SoftwareRevision $v1
"$id riportato alla software revision $v1" 

gruppo wpc-g-cli cancellato (se c'era)

gruppo wpc-g-docs cancellato (se c'era)

device-da-aggiornare riportato alla software revision 1.0.0

## Cosa abbiamo visto
- La **software revision** è una proprietà del device; un aggiornamento la cambia.
- Un **gruppo** è una query sulle proprietà, immutabile: il servizio vuole `*` o **una clausola `WHERE`** (`WHERE model = 'RobotV1' AND softwareRevision = '1.0.0'` funziona).
- Il gruppo è una **fotografia**: dopo il cambio della software revision conta ancora gli stessi membri, e il **refresh si può chiedere una volta all'ora**, anche subito dopo la creazione (`GroupRefreshRateLimited`).

Un job di aggiornamento vero, con un device che chiede e installa, è nei notebook 14c e 14d.

Prossimo notebook: **16**, un file firmato.